# Branch 2 — Textual Models — v8.1 Paper-Ready T4 x2


### v8 paper-ready corrections

- deduplicates overlapping text sources before building `sec_text`;
- audits MD&A/Risk Factors coverage and shows model-input samples;
- holds rows/splits fixed across section ablations;
- keeps TF-IDF + LR as a baseline;
- repeats the section ablation with the strongest validation-selected dense family;
- runs repeated shuffled-text controls;
- avoids calling every small score decrease “noise” without uncertainty.


In [ ]:
# SIC 3674 multimodal dataset input — Kaggle + Colab aware
from pathlib import Path

TARGET_DATASET_NAMES = [
    "sic3674_multimodal_model_rows.parquet",
    "sic3674_multimodal_model_rows.csv",
]

def resolve_sic3674_dataset_path():
    # 1) Kaggle attached datasets / working files
    for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
        if root.exists():
            for filename in TARGET_DATASET_NAMES:
                matches = list(root.rglob(filename))
                if matches:
                    return matches[0]

    # 2) Colab / Google Drive
    candidates = [
        Path("/content/drive/MyDrive/sic3674_output/sic3674_multimodal_model_rows.parquet"),
        Path("/content/drive/MyDrive/sic3674_output/sic3674_multimodal_model_rows.csv"),
        Path("/content/sic3674_output/sic3674_multimodal_model_rows.parquet"),
        Path("/content/sic3674_output/sic3674_multimodal_model_rows.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate sic3674_multimodal_model_rows.parquet/csv. "
        "Attach the dataset to Kaggle or place it in the Colab/Drive sic3674_output folder."
    )

SIC3674_DATA_PATH = resolve_sic3674_dataset_path()
print("Using SIC 3674 dataset:", SIC3674_DATA_PATH)


In [ ]:
!pip -q install pyarrow openpyxl xgboost sentence-transformers transformers accelerate beautifulsoup4 requests tqdm scipy

In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, f1_score, log_loss, precision_score,
    recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
DATA_PATH: Path | None = None
NEUTRAL_BAND = 0.02
C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0, 10.0]
THRESHOLD_GRID = np.linspace(0.20, 0.80, 121)


OUTPUT_DIR = Path(
    '/kaggle/working/datasets/pjbob1/semiconductor_branch_textual_v8_1_paper_ready_t4x2'
    if Path('/kaggle/working').exists()
    else '/content/semiconductor_branch_textual_v8_1_paper_ready_t4x2'
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DATA_PATH: Path | None = None
SEC_USER_AGENT = 'YOUR NAME your.email@example.com'
DOWNLOAD_SEC_TEXT_FROM_EDGAR = True
TEXT_EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
MIN_TEXT_CHARS = 500
MAX_DOCUMENT_CHARS = 120_000
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40
MAX_CHUNKS_PER_FILING = 16
TEXT_BATCH_SIZE = 32
TEXT_PCA_COMPONENTS = 32
TEXT_PCA_GRID = [16, 32, 64, 128]
RUN_LOCO = False
RUN_FINBERT_SENTIMENT = True
FINBERT_MODEL_NAME = 'ProsusAI/finbert'
TEXT_CACHE_DIR = OUTPUT_DIR/'sec_text_cache'
TEXT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# GPU diagnostics. On Kaggle T4 x2 this should report two CUDA devices.
try:
    import torch
    GPU_COUNT = torch.cuda.device_count() if torch.cuda.is_available() else 0
    GPU_DEVICES = [f"cuda:{i}" for i in range(GPU_COUNT)]
    print("CUDA available:", torch.cuda.is_available())
    print("GPU count:", GPU_COUNT)
    for i in range(GPU_COUNT):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")
    if GPU_COUNT >= 2:
        print("T4 x2 / multi-GPU mode enabled for text inference.")
    elif GPU_COUNT == 1:
        print("Single-GPU mode enabled.")
    else:
        print("No CUDA GPU detected; text inference will use CPU.")
except Exception as exc:
    GPU_COUNT = 0
    GPU_DEVICES = []
    print("GPU detection failed:", repr(exc))

## Load the same dataset used by v6.1

In [ ]:
def discover_data_path() -> Path:
    # Prefer the resolved SIC 3674 multimodal dataset.
    if "SIC3674_DATA_PATH" in globals() and Path(SIC3674_DATA_PATH).exists():
        return Path(SIC3674_DATA_PATH)

    candidates = [
        Path('/content/drive/MyDrive/sec_research_semiconductor/semiconductor_sec_numeric_text.parquet'),
        Path('/content/drive/MyDrive/sec_research_semiconductor/sec_experiment_semiconductor.parquet'),
        Path('/content/semiconductor_sec_numeric_text.parquet'),
        Path('/content/sec_experiment_semiconductor.parquet'),
        Path('semiconductor_sec_numeric_text.parquet'),
        Path('semiconductor_sec_numeric_text.csv'),
        Path('sec_experiment_semiconductor.parquet'),
        Path('model_dataset.csv'),
    ]
    for candidate in candidates:
        if candidate.exists():
            print('Found dataset automatically:', candidate)
            return candidate

    try:
        from google.colab import files
        print('Upload the same parquet/CSV dataset used by the experiment.')
        uploaded = files.upload()
        choices = [
            Path(name) for name in uploaded
            if Path(name).suffix.lower() in {'.parquet', '.csv'}
        ]
        if not choices:
            raise FileNotFoundError('Upload a .parquet or .csv file.')
        return choices[0]
    except ImportError as exc:
        raise FileNotFoundError(
            'Dataset not found. Attach it to Kaggle or set DATA_PATH explicitly.'
        ) from exc

resolved_path = DATA_PATH if DATA_PATH is not None else discover_data_path()
if resolved_path.suffix.lower() == '.parquet':
    raw = pd.read_parquet(resolved_path)
elif resolved_path.suffix.lower() == '.csv':
    raw = pd.read_csv(resolved_path)
else:
    raise ValueError('Use a .parquet or .csv dataset.')

print('Loaded:', resolved_path)
print('Rows:', len(raw), '| Columns:', len(raw.columns))


## Rebuild target exactly as v6.1

In [ ]:
data = raw.copy()

# Standardize identifiers and dates.
if "cik" not in data.columns:
    if "ticker" not in data.columns:
        raise ValueError("The dataset must contain either cik or ticker.")
    data["cik"] = data["ticker"].astype(str)

if "ticker" not in data.columns:
    data["ticker"] = data["cik"].astype(str)

date_candidates = [
    "quarter_end",
    "feature_cutoff_date",
    "label_available_date",
]
for column in date_candidates:
    if column in data.columns:
        data[column] = pd.to_datetime(data[column], errors="coerce")

if "quarter_end" not in data.columns:
    raise ValueError("The dataset must contain quarter_end.")

data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

# Map names from the earlier proof-of-concept dataset when needed.
rename_aliases = {
    "revenue_mm": "revenue",
    "inventory_mm": "inventory",
    "accounts_receivable_mm": "accounts_receivable",
    "operating_cash_flow_mm": "operating_cash_flow",
    "inventory_ratio": "inventory_to_ttm_revenue",
    "ar_ratio": "receivables_to_ttm_revenue",
    "ocf_margin": "cash_flow_margin",
}
for old_name, new_name in rename_aliases.items():
    if new_name not in data.columns and old_name in data.columns:
        data[new_name] = pd.to_numeric(data[old_name], errors="coerce")

numeric_candidates = [
    "revenue",
    "revenue_yoy_growth",
    "revenue_momentum",
    "next_quarter_growth",
    "gross_margin",
    "operating_margin",
    "cash_flow_margin",
    "inventory",
    "inventory_to_ttm_revenue",
    "accounts_receivable",
    "receivables_to_ttm_revenue",
    "capital_expenditures",
    "capex_to_revenue",
    "total_assets",
    "log_assets",
    "liabilities_to_assets",
]
for column in numeric_candidates:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

# Reconstruct YoY growth if it is missing and revenue is available.
if "revenue_yoy_growth" not in data.columns and "revenue" in data.columns:
    revenue_lag4 = grouped["revenue"].shift(4)
    quarter_lag4 = grouped["quarter_end"].shift(4)
    gap4 = (data["quarter_end"] - quarter_lag4).dt.days
    data["revenue_yoy_growth"] = np.where(
        gap4.between(320, 410),
        data["revenue"] / revenue_lag4 - 1.0,
        np.nan,
    )

# Reconstruct current momentum if missing.
if "revenue_momentum" not in data.columns:
    previous_growth = grouped["revenue_yoy_growth"].shift(1)
    previous_end = grouped["quarter_end"].shift(1)
    gap1 = (data["quarter_end"] - previous_end).dt.days
    data["revenue_momentum"] = np.where(
        gap1.between(60, 125),
        data["revenue_yoy_growth"] - previous_growth,
        np.nan,
    )

# Reconstruct next-quarter growth if missing.
if "next_quarter_growth" not in data.columns:
    next_growth = grouped["revenue_yoy_growth"].shift(-1)
    next_end = grouped["quarter_end"].shift(-1)
    next_gap = (next_end - data["quarter_end"]).dt.days
    data["next_quarter_growth"] = np.where(
        next_gap.between(60, 125),
        next_growth,
        np.nan,
    )

data["future_growth_change"] = (
    data["next_quarter_growth"] - data["revenue_yoy_growth"]
)

data["target_clean"] = pd.Series(
    np.select(
        [
            data["future_growth_change"] > NEUTRAL_BAND,
            data["future_growth_change"] < -NEUTRAL_BAND,
        ],
        [1.0, 0.0],
        default=np.nan,
    ),
    index=data.index,
)

data["target_status"] = np.select(
    [
        data["future_growth_change"] > NEUTRAL_BAND,
        data["future_growth_change"] < -NEUTRAL_BAND,
        data["future_growth_change"].abs() <= NEUTRAL_BAND,
    ],
    ["Accelerating", "Decelerating", "Neutral"],
    default="Unavailable",
)

target_summary = (
    data["target_status"]
    .value_counts(dropna=False)
    .rename_axis("target_status")
    .reset_index(name="rows")
)
target_summary["fraction"] = target_summary["rows"] / len(data)
display(target_summary)

## Preserve v6.1 feature engineering/split timing

In [ ]:
data = data.sort_values(["cik", "quarter_end"]).reset_index(drop=True)
grouped = data.groupby("cik", group_keys=False)

previous_end = grouped["quarter_end"].shift(1)
gap1 = (data["quarter_end"] - previous_end).dt.days
valid_qoq = gap1.between(60, 125)

quarter_lag4 = grouped["quarter_end"].shift(4)
gap4 = (data["quarter_end"] - quarter_lag4).dt.days
valid_yoy = gap4.between(320, 410)

def add_qoq_change(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(1)
        data[output] = np.where(valid_qoq, data[column] - lagged, np.nan)

def add_yoy_growth(column: str, output: str) -> None:
    if column in data.columns:
        lagged = grouped[column].shift(4)
        data[output] = np.where(
            valid_yoy & (lagged.abs() > 1e-12),
            data[column] / lagged - 1.0,
            np.nan,
        )

if "revenue" in data.columns:
    revenue_lag1 = grouped["revenue"].shift(1)
    data["sequential_revenue_growth"] = np.where(
        valid_qoq & (revenue_lag1.abs() > 1e-12),
        data["revenue"] / revenue_lag1 - 1.0,
        np.nan,
    )

add_qoq_change("gross_margin", "gross_margin_change")
add_qoq_change("operating_margin", "operating_margin_change")
add_qoq_change("cash_flow_margin", "cash_flow_margin_change")
add_qoq_change(
    "inventory_to_ttm_revenue",
    "inventory_to_ttm_revenue_change",
)
add_qoq_change(
    "receivables_to_ttm_revenue",
    "receivables_to_ttm_revenue_change",
)
add_qoq_change("capex_to_revenue", "capex_to_revenue_change")

add_yoy_growth("inventory", "inventory_yoy_growth")
add_yoy_growth("accounts_receivable", "receivables_yoy_growth")
add_yoy_growth("capital_expenditures", "capex_yoy_growth")

if "inventory_yoy_growth" in data.columns:
    data["inventory_revenue_growth_gap"] = (
        data["inventory_yoy_growth"] - data["revenue_yoy_growth"]
    )

if "receivables_yoy_growth" in data.columns:
    data["receivables_revenue_growth_gap"] = (
        data["receivables_yoy_growth"] - data["revenue_yoy_growth"]
    )

data["calendar_quarter"] = data["quarter_end"].dt.to_period("Q")

relative_base_features = [
    column
    for column in [
        "revenue_yoy_growth",
        "revenue_momentum",
        "sequential_revenue_growth",
        "gross_margin",
        "gross_margin_change",
        "cash_flow_margin",
        "inventory_to_ttm_revenue",
        "inventory_revenue_growth_gap",
        "receivables_to_ttm_revenue",
        "capex_to_revenue",
    ]
    if column in data.columns
]

quarter_medians = (
    data.groupby("calendar_quarter")[relative_base_features]
    .median()
    .sort_index()
)
prior_quarter_medians = quarter_medians.shift(1).add_suffix(
    "_prior_sector_median"
)

data = data.merge(
    prior_quarter_medians,
    left_on="calendar_quarter",
    right_index=True,
    how="left",
)

for feature in relative_base_features:
    median_column = f"{feature}_prior_sector_median"
    data[f"{feature}_relative_to_sector"] = (
        data[feature] - data[median_column]
    )

engineered_features = [
    column
    for column in data.columns
    if (
        column.endswith("_change")
        or column.endswith("_yoy_growth")
        or column.endswith("_growth_gap")
        or column.endswith("_relative_to_sector")
        or column == "sequential_revenue_growth"
    )
]

print("Engineered features:", len(engineered_features))
display(data[["ticker", "quarter_end"] + engineered_features[:12]].head(10))

In [ ]:
# Use an existing split only when it contains all three required groups.
required_splits = {"train", "validation", "test"}

existing_splits = (
    set(data["split"].dropna().astype(str).str.lower().unique())
    if "split" in data.columns
    else set()
)

existing_split_is_usable = required_splits.issubset(existing_splits)

if existing_split_is_usable:
    data["split"] = data["split"].astype(str).str.lower()
    print("Using the existing train/validation/test split.")
else:
    if "split" in data.columns:
        print(
            "The existing split column does not contain all three groups. "
            "Rebuilding the split chronologically."
        )

    data["split"] = pd.NA

    # Use the most conservative available timestamp:
    # when the target became knowable, then the feature cutoff, then quarter end.
    if (
        "label_available_date" in data.columns
        and data["label_available_date"].notna().any()
    ):
        split_date_column = "label_available_date"
    elif (
        "feature_cutoff_date" in data.columns
        and data["feature_cutoff_date"].notna().any()
    ):
        split_date_column = "feature_cutoff_date"
    else:
        split_date_column = "quarter_end"

    eligible_dates = (
        data.loc[data["target_clean"].notna(), split_date_column]
        .dropna()
        .drop_duplicates()
        .sort_values()
        .reset_index(drop=True)
    )

    if len(eligible_dates) < 3:
        raise ValueError(
            "At least three distinct dated periods are required to create "
            "train, validation, and test splits."
        )

    # Automatic chronological 60% / 20% / 20% split by distinct dates.
    train_position = max(0, min(len(eligible_dates) - 3, int(len(eligible_dates) * 0.60) - 1))
    validation_position = max(
        train_position + 1,
        min(len(eligible_dates) - 2, int(len(eligible_dates) * 0.80) - 1),
    )

    automatic_train_end = eligible_dates.iloc[train_position]
    automatic_validation_end = eligible_dates.iloc[validation_position]

    labeled = data["target_clean"].notna()
    split_dates = data[split_date_column]

    data.loc[
        labeled & (split_dates <= automatic_train_end),
        "split",
    ] = "train"

    data.loc[
        labeled
        & (split_dates > automatic_train_end)
        & (split_dates <= automatic_validation_end),
        "split",
    ] = "validation"

    data.loc[
        labeled & (split_dates > automatic_validation_end),
        "split",
    ] = "test"

    print("Split date column:", split_date_column)
    print("Automatic train end:", automatic_train_end)
    print("Automatic validation end:", automatic_validation_end)

model_data = data[
    data["target_clean"].notna()
    & data["split"].isin(["train", "validation", "test"])
].copy()
model_data["target_clean"] = model_data["target_clean"].astype(int)

split_summary = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        companies=("cik", "nunique"),
        first_quarter=("quarter_end", "min"),
        last_quarter=("quarter_end", "max"),
        acceleration_rate=("target_clean", "mean"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
display(split_summary)

available_splits = set(model_data["split"].dropna().unique())
missing_splits = required_splits - available_splits
if missing_splits:
    raise ValueError(
        f"Could not create these splits: {sorted(missing_splits)}. "
        "The dataset may have too few labeled dates after applying the "
        "neutral band."
    )

for split_name in ["train", "validation", "test"]:
    split_frame = model_data[model_data["split"] == split_name]
    if split_frame["target_clean"].nunique() < 2:
        print(
            f"Warning: {split_name} contains only one target class after "
            f"applying the {NEUTRAL_BAND:.1%} neutral band. "
            "Try reducing NEUTRAL_BAND to 0.01 if model fitting fails."
        )

## Attach SEC filing text

In [ ]:
import hashlib
import html as html_lib
import json
import re
import time
from collections.abc import Iterable

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm


TEXT_COLUMN_CANDIDATES = [
    "sec_text",
    "filing_text",
    "document_text",
    "mda_text",
    "management_discussion_text",
    "risk_factors_text",
    "risk_text",
]


def normalize_accession(value: object) -> str | None:
    if pd.isna(value):
        return None
    text = str(value).strip()
    return text if text else None


def _deduplicate_text_pieces(values: list[str]) -> list[str]:
    cleaned = []
    seen = set()
    for value in values:
        text = re.sub(r"\s+", " ", str(value or "")).strip()
        if not text:
            continue
        key = text.lower()
        if key in seen:
            continue
        seen.add(key)
        cleaned.append(text)

    cleaned = sorted(cleaned, key=len, reverse=True)
    kept = []
    for text in cleaned:
        lower = text.lower()
        if any(lower in existing.lower() for existing in kept):
            continue
        kept.append(text)
    return kept


def combine_available_text_columns(frame: pd.DataFrame) -> pd.Series:
    available = [
        column for column in TEXT_COLUMN_CANDIDATES
        if column in frame.columns
    ]
    if not available:
        return pd.Series("", index=frame.index, dtype="object")

    def build_row(row):
        pieces = _deduplicate_text_pieces([
            row[column]
            for column in available
            if pd.notna(row[column])
        ])
        return "\n\n".join(pieces)

    return frame[available].apply(build_row, axis=1)


def discover_text_data_path() -> Path | None:
    candidates = [
        Path("/content/drive/MyDrive/sec_research_large_150/sec_filing_text.parquet"),
        Path("/content/drive/MyDrive/sec_research_semiconductor/sec_filing_text.parquet"),
        Path("/content/sec_research_large_150/sec_filing_text.parquet"),
        Path("/content/sec_research_semiconductor/sec_filing_text.parquet"),
        Path("sec_filing_text.parquet"),
        Path("sec_filing_text.csv"),
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def read_table(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError("Text dataset must be a .parquet or .csv file.")


def merge_external_text(
    base: pd.DataFrame,
    text_frame: pd.DataFrame,
) -> pd.DataFrame:
    external = text_frame.copy()

    for frame in [base, external]:
        if "quarter_end" in frame.columns:
            frame["quarter_end"] = pd.to_datetime(
                frame["quarter_end"], errors="coerce"
            )
        if "filing_date" in frame.columns:
            frame["filing_date"] = pd.to_datetime(
                frame["filing_date"], errors="coerce"
            )
        if "accession_number" in frame.columns:
            frame["accession_number"] = frame[
                "accession_number"
            ].map(normalize_accession)
        if "cik" in frame.columns:
            frame["cik"] = frame["cik"].astype(str)
        if "ticker" in frame.columns:
            frame["ticker"] = frame["ticker"].astype(str)

    external["external_sec_text"] = combine_available_text_columns(
        external
    )

    if "accession_number" in base.columns and "accession_number" in external.columns:
        merge_keys = ["accession_number"]
    elif all(
        column in base.columns and column in external.columns
        for column in ["cik", "quarter_end"]
    ):
        merge_keys = ["cik", "quarter_end"]
    elif all(
        column in base.columns and column in external.columns
        for column in ["ticker", "quarter_end"]
    ):
        merge_keys = ["ticker", "quarter_end"]
    else:
        raise ValueError(
            "The text dataset must share accession_number, "
            "(cik, quarter_end), or (ticker, quarter_end) with the "
            "experiment dataset."
        )

    keep_columns = merge_keys + ["external_sec_text"]
    if "filing_date" in external.columns:
        keep_columns.append("filing_date")

    external = (
        external[keep_columns]
        .drop_duplicates(subset=merge_keys, keep="last")
    )

    merged = base.merge(
        external,
        on=merge_keys,
        how="left",
        suffixes=("", "_text_source"),
    )

    merged["sec_text"] = np.where(
        merged["sec_text"].fillna("").str.len()
        >= merged["external_sec_text"].fillna("").str.len(),
        merged["sec_text"].fillna(""),
        merged["external_sec_text"].fillna(""),
    )
    return merged


def clean_filing_html(raw_html: str) -> str:
    soup = BeautifulSoup(raw_html, "html.parser")

    for tag in soup(["script", "style", "noscript", "svg"]):
        tag.decompose()

    # The project is using the narrative writing; large XBRL tables add
    # many repeated numbers and labels without much prose.
    for table in soup.find_all("table"):
        table.decompose()

    text = soup.get_text(" ")
    text = html_lib.unescape(text)
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def extract_longest_section(
    text: str,
    start_patterns: list[str],
    end_patterns: list[str],
    minimum_chars: int = 500,
    maximum_chars: int = 100_000,
) -> str:
    starts = []
    for pattern in start_patterns:
        starts.extend(re.finditer(pattern, text, flags=re.I))

    ends = []
    for pattern in end_patterns:
        ends.extend(re.finditer(pattern, text, flags=re.I))

    candidates: list[str] = []
    for start in starts:
        possible_ends = [
            end for end in ends
            if end.start() > start.end() + minimum_chars
        ]
        if not possible_ends:
            continue
        end = min(possible_ends, key=lambda match: match.start())
        candidate = text[start.start():end.start()].strip()
        if minimum_chars <= len(candidate) <= maximum_chars:
            candidates.append(candidate)

    return max(candidates, key=len) if candidates else ""


def extract_narrative_sections(clean_text: str) -> str:
    mda = extract_longest_section(
        clean_text,
        start_patterns=[
            r"\bitem\s+7[\.\:\-\s]+management[’']?s?\s+discussion",
            r"\bitem\s+2[\.\:\-\s]+management[’']?s?\s+discussion",
        ],
        end_patterns=[
            r"\bitem\s+7a[\.\:\-\s]+",
            r"\bitem\s+8[\.\:\-\s]+financial",
            r"\bitem\s+3[\.\:\-\s]+quantitative",
            r"\bitem\s+4[\.\:\-\s]+controls",
        ],
    )

    risks = extract_longest_section(
        clean_text,
        start_patterns=[
            r"\bitem\s+1a[\.\:\-\s]+risk\s+factors",
        ],
        end_patterns=[
            r"\bitem\s+1b[\.\:\-\s]+",
            r"\bitem\s+1c[\.\:\-\s]+",
            r"\bitem\s+2[\.\:\-\s]+",
        ],
    )

    sections = []
    if mda:
        sections.append("MANAGEMENT DISCUSSION AND ANALYSIS\n" + mda)
    if risks:
        sections.append("RISK FACTORS\n" + risks)

    if sections:
        return "\n\n".join(sections)[:MAX_DOCUMENT_CHARS]

    # Fallback when filing headings differ from the standard patterns.
    return clean_text[:MAX_DOCUMENT_CHARS]


def valid_sec_user_agent(user_agent: str) -> bool:
    lowered = user_agent.lower()
    return (
        "your name" not in lowered
        and "example.com" not in lowered
        and "@" in user_agent
        and len(user_agent.strip()) >= 8
    )


class EdgarTextDownloader:
    def __init__(self, user_agent: str, cache_dir: Path):
        if not valid_sec_user_agent(user_agent):
            raise ValueError(
                "Replace SEC_USER_AGENT with your real name and email "
                "before downloading from EDGAR."
            )

        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": user_agent,
                "Accept-Encoding": "gzip, deflate",
            }
        )
        self.cache_dir = cache_dir
        self.cache_dir.mkdir(parents=True, exist_ok=True)
        self.submission_cache: dict[str, dict[str, str]] = {}

    def get_json(self, url: str) -> dict:
        response = self.session.get(url, timeout=60)
        response.raise_for_status()
        time.sleep(0.20)
        return response.json()

    def get_text(self, url: str) -> str:
        response = self.session.get(url, timeout=90)
        response.raise_for_status()
        time.sleep(0.20)
        return response.text

    def _add_submission_rows(
        self,
        mapping: dict[str, str],
        payload: dict,
    ) -> None:
        accessions = payload.get("accessionNumber", [])
        primary_documents = payload.get("primaryDocument", [])
        for accession, document in zip(
            accessions, primary_documents, strict=False
        ):
            if accession and document:
                mapping[str(accession)] = str(document)

    def submission_map(self, cik: str) -> dict[str, str]:
        cik_key = str(int(float(cik))).zfill(10)
        if cik_key in self.submission_cache:
            return self.submission_cache[cik_key]

        payload = self.get_json(
            f"https://data.sec.gov/submissions/CIK{cik_key}.json"
        )
        mapping: dict[str, str] = {}
        self._add_submission_rows(
            mapping,
            payload.get("filings", {}).get("recent", {}),
        )

        # Older filings can be stored in additional submission JSON files.
        for file_info in payload.get("filings", {}).get("files", []):
            file_name = file_info.get("name")
            if not file_name:
                continue
            old_payload = self.get_json(
                f"https://data.sec.gov/submissions/{file_name}"
            )
            self._add_submission_rows(mapping, old_payload)

        self.submission_cache[cik_key] = mapping
        return mapping

    def primary_document(
        self,
        cik: str,
        accession: str,
        supplied_document: object = None,
    ) -> str:
        if supplied_document is not None and not pd.isna(supplied_document):
            supplied = str(supplied_document).strip()
            if supplied:
                return supplied

        mapping = self.submission_map(cik)
        document = mapping.get(accession)
        if document:
            return document

        # Final fallback: choose the largest non-index HTML document.
        cik_number = str(int(float(cik)))
        accession_compact = accession.replace("-", "")
        index_payload = self.get_json(
            "https://www.sec.gov/Archives/edgar/data/"
            f"{cik_number}/{accession_compact}/index.json"
        )
        items = (
            index_payload.get("directory", {}).get("item", [])
        )
        html_items = [
            item for item in items
            if str(item.get("name", "")).lower().endswith(
                (".htm", ".html")
            )
            and "-index." not in str(item.get("name", "")).lower()
            and "filingsummary" not in str(item.get("name", "")).lower()
        ]
        if not html_items:
            raise FileNotFoundError(
                f"No primary HTML document found for {accession}."
            )
        chosen = max(
            html_items,
            key=lambda item: int(item.get("size", 0) or 0),
        )
        return str(chosen["name"])

    def filing_text(
        self,
        cik: str,
        accession: str,
        supplied_document: object = None,
    ) -> str:
        accession = normalize_accession(accession)
        if accession is None:
            return ""

        cache_path = self.cache_dir / f"{accession}.txt"
        if cache_path.exists():
            return cache_path.read_text(
                encoding="utf-8", errors="ignore"
            )

        document = self.primary_document(
            cik, accession, supplied_document
        )
        cik_number = str(int(float(cik)))
        accession_compact = accession.replace("-", "")
        url = (
            "https://www.sec.gov/Archives/edgar/data/"
            f"{cik_number}/{accession_compact}/{document}"
        )
        raw_html = self.get_text(url)
        cleaned = clean_filing_html(raw_html)
        narrative = extract_narrative_sections(cleaned)
        cache_path.write_text(narrative, encoding="utf-8")
        return narrative


# Begin with any text already present in the experiment dataset.
model_data = model_data.copy()
model_data["sec_text"] = combine_available_text_columns(model_data)

# Merge a separate text table when configured or automatically found.
resolved_text_path = (
    TEXT_DATA_PATH
    if TEXT_DATA_PATH is not None
    else discover_text_data_path()
)
if resolved_text_path is not None:
    if not resolved_text_path.exists():
        raise FileNotFoundError(
            f"TEXT_DATA_PATH does not exist: {resolved_text_path}"
        )
    external_text = read_table(resolved_text_path)
    model_data = merge_external_text(model_data, external_text)
    print("Merged SEC text dataset:", resolved_text_path)

# Download only missing filing text, using one request per unique filing.
missing_text = model_data["sec_text"].fillna("").str.len() < MIN_TEXT_CHARS
can_download = (
    DOWNLOAD_SEC_TEXT_FROM_EDGAR
    and missing_text.any()
    and {"cik", "accession_number"}.issubset(model_data.columns)
    and valid_sec_user_agent(SEC_USER_AGENT)
)

if can_download:
    downloader = EdgarTextDownloader(
        SEC_USER_AGENT,
        TEXT_CACHE_DIR / "filings",
    )

    unique_filings = (
        model_data.loc[
            missing_text,
            [
                column
                for column in [
                    "cik",
                    "accession_number",
                    "primary_document",
                ]
                if column in model_data.columns
            ],
        ]
        .dropna(subset=["cik", "accession_number"])
        .drop_duplicates(subset=["cik", "accession_number"])
    )

    downloaded_text: dict[tuple[str, str], str] = {}
    failures = []

    for _, filing in tqdm(
        unique_filings.iterrows(),
        total=len(unique_filings),
        desc="Downloading SEC filings",
    ):
        cik = str(filing["cik"])
        accession = normalize_accession(filing["accession_number"])
        if accession is None:
            continue
        supplied_document = (
            filing.get("primary_document")
            if "primary_document" in filing.index
            else None
        )
        try:
            downloaded_text[(cik, accession)] = (
                downloader.filing_text(
                    cik,
                    accession,
                    supplied_document,
                )
            )
        except Exception as exc:
            failures.append(
                {
                    "cik": cik,
                    "accession_number": accession,
                    "error": str(exc),
                }
            )

    row_keys = list(
        zip(
            model_data["cik"].astype(str),
            model_data["accession_number"].map(normalize_accession),
        )
    )
    downloaded_series = pd.Series(
        [
            downloaded_text.get(key, "")
            for key in row_keys
        ],
        index=model_data.index,
    )
    replace_mask = (
        model_data["sec_text"].fillna("").str.len()
        < downloaded_series.str.len()
    )
    model_data.loc[replace_mask, "sec_text"] = downloaded_series[
        replace_mask
    ]

    if failures:
        failure_frame = pd.DataFrame(failures)
        failure_frame.to_csv(
            TEXT_CACHE_DIR / "edgar_download_failures.csv",
            index=False,
        )
        print(
            f"{len(failures)} filing downloads failed. Details were saved "
            "to edgar_download_failures.csv."
        )

elif missing_text.any():
    print(
        "SEC text is not yet available for all rows.\n"
        "Use one of these options:\n"
        "  1. Put sec_text, filing_text, mda_text, or risk_factors_text "
        "in the experiment dataset.\n"
        "  2. Set TEXT_DATA_PATH to a matching text parquet/CSV.\n"
        "  3. Replace SEC_USER_AGENT with your real name and email so "
        "the notebook can download filings from EDGAR."
    )

# Prevent obvious timing leakage when both timestamps are available.
if (
    "filing_date" in model_data.columns
    and "feature_cutoff_date" in model_data.columns
):
    filing_dates = pd.to_datetime(
        model_data["filing_date"], errors="coerce"
    )
    cutoff_dates = pd.to_datetime(
        model_data["feature_cutoff_date"], errors="coerce"
    )
    late_text = (
        filing_dates.notna()
        & cutoff_dates.notna()
        & (filing_dates > cutoff_dates)
    )
    if late_text.any():
        print(
            f"Removed text from {late_text.sum()} rows because the filing "
            "date was after the feature cutoff date."
        )
        model_data.loc[late_text, "sec_text"] = ""

model_data["text_characters"] = (
    model_data["sec_text"].fillna("").str.len()
)
model_data["has_sec_text"] = (
    model_data["text_characters"] >= MIN_TEXT_CHARS
)

text_coverage = (
    model_data.groupby("split")
    .agg(
        rows=("target_clean", "size"),
        rows_with_text=("has_sec_text", "sum"),
        median_text_characters=("text_characters", "median"),
    )
    .reindex(["train", "validation", "test"])
    .reset_index()
)
text_coverage["coverage"] = (
    text_coverage["rows_with_text"] / text_coverage["rows"]
)
display(text_coverage)

## Keep comparable rows with filing text

In [ ]:
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer

text_model_data = model_data[
    model_data['has_sec_text'] & model_data['split'].isin(['train','validation','test'])
].copy()
for s in ['train','validation','test']:
    sub = text_model_data[text_model_data.split == s]
    if sub.empty or sub.target_clean.nunique() < 2:
        raise ValueError(f'{s} needs text rows from both classes. Add more text/data or reduce NEUTRAL_BAND.')
print(text_model_data.groupby('split').agg(rows=('target_clean','size'), companies=('cik','nunique'), classes=('target_clean','nunique')))
TEXT_MODELS_READY = True

## v8 text-content audit


In [ ]:
source_columns_present = [
    c for c in TEXT_COLUMN_CANDIDATES if c in model_data.columns
]
text_source_audit = []
for column in source_columns_present:
    values = model_data[column].fillna("").astype(str)
    nonempty = values.str.len() > 0
    text_source_audit.append({
        "source_column": column,
        "rows_nonempty": int(nonempty.sum()),
        "rows_ge_500_chars": int((values.str.len() >= 500).sum()),
        "median_chars_nonempty": (
            float(values.loc[nonempty].str.len().median())
            if nonempty.any() else 0.0
        ),
    })
text_source_audit = pd.DataFrame(text_source_audit)
display(text_source_audit)

containment_rows = []
if "sec_text" in model_data.columns:
    for section_col in [
        "mda_text", "management_discussion_text",
        "risk_factors_text", "risk_text",
    ]:
        if section_col not in model_data.columns:
            continue
        section = model_data[section_col].fillna("").astype(str)
        full = model_data["sec_text"].fillna("").astype(str)
        eligible = section.str.len() >= 300
        count = 0
        for sec, whole in zip(section.loc[eligible], full.loc[eligible]):
            count += int(sec.lower() in whole.lower())
        containment_rows.append({
            "section": section_col,
            "eligible_rows": int(eligible.sum()),
            "rows_section_already_in_sec_text": count,
            "containment_rate": (
                float(count/eligible.sum()) if eligible.sum() else np.nan
            ),
        })
text_containment_audit = pd.DataFrame(containment_rows)
display(text_containment_audit)

sample_columns = [c for c in [
    "cik","ticker","quarter_end",
    "mda_text","risk_factors_text","sec_text"
] if c in text_model_data.columns]
text_samples = text_model_data[sample_columns].head(5).copy()
for c in ["mda_text","risk_factors_text","sec_text"]:
    if c in text_samples.columns:
        text_samples[c] = text_samples[c].fillna("").astype(str).str.slice(0,1000)
display(text_samples)

text_source_audit.to_csv(OUTPUT_DIR/"text_source_audit_v8.csv", index=False)
text_containment_audit.to_csv(OUTPUT_DIR/"text_containment_audit_v8.csv", index=False)


## Sentence Transformer embeddings

### GPU acceleration (Kaggle T4 x2)

- Sentence Transformer encoding uses both CUDA devices when two GPUs are available.
- FinBERT creates one model copy per GPU and splits text chunks across the GPUs concurrently.
- GPU inference uses FP16 on CUDA for faster T4 Tensor Core execution.
- Sentence embeddings and FinBERT features are cached under `TEXT_CACHE_DIR` so reruns avoid repeated neural inference.
- TF-IDF, logistic regression, PCA, metrics, and threshold tuning remain CPU/scikit-learn operations.


In [ ]:
def chunk_document(
    text: str,
    chunk_words: int = CHUNK_WORDS,
    overlap_words: int = CHUNK_OVERLAP_WORDS,
    maximum_chunks: int = MAX_CHUNKS_PER_FILING,
) -> list[str]:
    words = str(text).split()
    if not words:
        return []

    step = max(1, chunk_words - overlap_words)
    chunks = [
        " ".join(words[start:start + chunk_words])
        for start in range(0, len(words), step)
        if len(words[start:start + chunk_words]) >= 30
    ]

    if not chunks:
        return [" ".join(words)]

    if len(chunks) > maximum_chunks:
        selected_indices = np.linspace(
            0,
            len(chunks) - 1,
            maximum_chunks,
            dtype=int,
        )
        chunks = [chunks[index] for index in selected_indices]

    return chunks


def text_hash(text: str) -> str:
    payload = (
        TEXT_EMBEDDING_MODEL
        + "\n"
        + str(MAX_CHUNKS_PER_FILING)
        + "\n"
        + str(text)
    )
    return hashlib.sha1(
        payload.encode("utf-8", errors="ignore")
    ).hexdigest()


def get_torch_devices() -> list[str]:
    """Return all CUDA devices when available, otherwise CPU."""
    import torch

    if torch.cuda.is_available():
        return [
            f"cuda:{index}"
            for index in range(torch.cuda.device_count())
        ]
    return ["cpu"]


def build_sentence_embeddings(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    from sentence_transformers import SentenceTransformer

    working = frame.copy()
    working["text_hash"] = working["sec_text"].map(text_hash)

    safe_model_name = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        TEXT_EMBEDDING_MODEL,
    )
    cache_path = (
        TEXT_CACHE_DIR
        / f"document_embeddings_{safe_model_name}.parquet"
    )

    cached = pd.DataFrame()
    if cache_path.exists():
        cached = pd.read_parquet(cache_path)

    cached_hashes = (
        set(cached["text_hash"])
        if not cached.empty and "text_hash" in cached.columns
        else set()
    )

    unique_documents = (
        working[["text_hash", "sec_text"]]
        .drop_duplicates("text_hash")
    )
    missing_documents = unique_documents[
        ~unique_documents["text_hash"].isin(cached_hashes)
    ]

    if not missing_documents.empty:
        devices = get_torch_devices()
        print("Sentence Transformer device(s):", devices)

        # Sentence Transformers supports a list such as
        # ["cuda:0", "cuda:1"] for multi-process multi-GPU encoding.
        encoder = SentenceTransformer(TEXT_EMBEDDING_MODEL)

        flat_chunks: list[str] = []
        owners: list[str] = []
        for row in missing_documents.itertuples(index=False):
            chunks = chunk_document(row.sec_text)
            flat_chunks.extend(chunks)
            owners.extend([row.text_hash] * len(chunks))

        if not flat_chunks:
            raise ValueError("No valid text chunks were produced.")

        encode_device = devices if len(devices) > 1 else devices[0]

        try:
            chunk_embeddings = encoder.encode(
                flat_chunks,
                batch_size=TEXT_BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
                device=encode_device,
            )
        except Exception as exc:
            # Multi-process GPU startup can occasionally fail in a notebook
            # runtime. Fall back to the first GPU rather than losing the run.
            if len(devices) <= 1:
                raise
            print(
                "Multi-GPU Sentence Transformer failed; "
                "falling back to cuda:0. Error:",
                repr(exc),
            )
            encoder = SentenceTransformer(
                TEXT_EMBEDDING_MODEL,
                device=devices[0],
            )
            chunk_embeddings = encoder.encode(
                flat_chunks,
                batch_size=TEXT_BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
                device=devices[0],
            )

        chunk_frame = pd.DataFrame(chunk_embeddings)
        chunk_frame.insert(0, "text_hash", owners)

        document_embeddings = (
            chunk_frame.groupby("text_hash", sort=False)
            .mean()
            .reset_index()
        )

        embedding_columns = [
            column
            for column in document_embeddings.columns
            if column != "text_hash"
        ]
        matrix = document_embeddings[embedding_columns].to_numpy()
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        matrix = matrix / np.maximum(norms, 1e-12)
        document_embeddings[embedding_columns] = matrix

        if cached.empty:
            cached = document_embeddings
        else:
            cached = pd.concat(
                [cached, document_embeddings],
                ignore_index=True,
            ).drop_duplicates("text_hash", keep="last")

        cached.to_parquet(cache_path, index=False)

    embedding_columns_in_cache = [
        column for column in cached.columns
        if column != "text_hash"
    ]
    renamed = {
        column: f"text_embedding_{int(column):03d}"
        for column in embedding_columns_in_cache
    }
    cached = cached.rename(columns=renamed)
    embedding_columns = list(renamed.values())

    working = working.merge(
        cached,
        on="text_hash",
        how="left",
    )
    return working, embedding_columns


if TEXT_MODELS_READY:
    text_model_data, text_embedding_columns = (
        build_sentence_embeddings(text_model_data)
    )
    print(
        "Sentence embedding dimensions:",
        len(text_embedding_columns),
    )
else:
    text_embedding_columns = []

## FinBERT financial-sentiment features

In [ ]:
def finbert_text_hash(text: str) -> str:
    payload = (
        FINBERT_MODEL_NAME
        + "\n180\n30\n12\n"
        + str(text)
    )
    return hashlib.sha1(
        payload.encode("utf-8", errors="ignore")
    ).hexdigest()


def _finbert_infer_chunks(
    chunks: list[str],
    indices: np.ndarray,
    device: str,
) -> tuple[np.ndarray, np.ndarray]:
    """Run one independent FinBERT worker on one GPU."""
    import torch
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
    )

    tokenizer = AutoTokenizer.from_pretrained(FINBERT_MODEL_NAME)

    model_kwargs = {}
    if device.startswith("cuda"):
        # T4 Tensor Cores are much faster with FP16 inference.
        model_kwargs["torch_dtype"] = torch.float16

    model = AutoModelForSequenceClassification.from_pretrained(
        FINBERT_MODEL_NAME,
        **model_kwargs,
    )
    model.to(device)
    model.eval()

    probability_batches = []
    for start in range(0, len(indices), TEXT_BATCH_SIZE):
        batch_indices = indices[start:start + TEXT_BATCH_SIZE]
        batch = [chunks[int(i)] for i in batch_indices]

        tokens = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        ).to(device)

        with torch.inference_mode():
            if device.startswith("cuda"):
                with torch.autocast(
                    device_type="cuda",
                    dtype=torch.float16,
                ):
                    logits = model(**tokens).logits
            else:
                logits = model(**tokens).logits

            batch_probabilities = torch.softmax(
                logits,
                dim=-1,
            ).float().cpu().numpy()

        probability_batches.append(batch_probabilities)

    if not probability_batches:
        return indices, np.empty((0, model.config.num_labels))

    return indices, np.vstack(probability_batches)


def build_finbert_sentiment_features(
    frame: pd.DataFrame,
) -> tuple[pd.DataFrame, list[str]]:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    from transformers import AutoConfig

    output = frame.copy()
    feature_names = [
        "finbert_positive_mean",
        "finbert_neutral_mean",
        "finbert_negative_mean",
        "finbert_negative_max",
        "finbert_negative_std",
        "finbert_positive_minus_negative",
    ]

    if not RUN_FINBERT_SENTIMENT:
        for feature in feature_names:
            output[feature] = 0.0
        return output, []

    output["finbert_text_hash"] = output["sec_text"].map(
        finbert_text_hash
    )
    cache_path = TEXT_CACHE_DIR / "finbert_sentiment_features.parquet"

    cached = pd.DataFrame()
    if cache_path.exists():
        cached = pd.read_parquet(cache_path)

    cached_hashes = (
        set(cached["finbert_text_hash"])
        if not cached.empty and "finbert_text_hash" in cached.columns
        else set()
    )

    unique_documents = (
        output[["finbert_text_hash", "sec_text"]]
        .drop_duplicates("finbert_text_hash")
    )
    missing_documents = unique_documents[
        ~unique_documents["finbert_text_hash"].isin(cached_hashes)
    ].reset_index(drop=True)

    if not missing_documents.empty:
        devices = get_torch_devices()
        print("FinBERT device(s):", devices)

        config = AutoConfig.from_pretrained(FINBERT_MODEL_NAME)
        id_to_label = {
            int(index): str(label).lower()
            for index, label in config.id2label.items()
        }
        num_labels = len(id_to_label)

        flat_chunks: list[str] = []
        owners: list[int] = []

        for doc_index, row in enumerate(
            missing_documents.itertuples(index=False)
        ):
            chunks = chunk_document(
                row.sec_text,
                chunk_words=180,
                overlap_words=30,
                maximum_chunks=12,
            )
            if not chunks:
                continue
            flat_chunks.extend(chunks)
            owners.extend([doc_index] * len(chunks))

        if not flat_chunks:
            raise ValueError("No valid FinBERT text chunks were produced.")

        all_indices = np.arange(len(flat_chunks), dtype=int)
        index_splits = [
            split.astype(int)
            for split in np.array_split(all_indices, len(devices))
            if len(split) > 0
        ]

        probability_matrix = np.empty(
            (len(flat_chunks), num_labels),
            dtype=np.float32,
        )

        if len(index_splits) == 1:
            indices, probs = _finbert_infer_chunks(
                flat_chunks,
                index_splits[0],
                devices[0],
            )
            probability_matrix[indices] = probs
        else:
            # One independent model copy per T4. Each GPU processes a
            # different half of the chunks concurrently.
            with ThreadPoolExecutor(
                max_workers=len(index_splits)
            ) as executor:
                futures = []
                for worker_index, indices in enumerate(index_splits):
                    device = devices[worker_index]
                    futures.append(
                        executor.submit(
                            _finbert_infer_chunks,
                            flat_chunks,
                            indices,
                            device,
                        )
                    )

                for future in as_completed(futures):
                    indices, probs = future.result()
                    probability_matrix[indices] = probs

        owners_array = np.asarray(owners, dtype=int)
        feature_rows = []

        for doc_index, row in enumerate(
            missing_documents.itertuples(index=False)
        ):
            doc_mask = owners_array == doc_index
            matrix = probability_matrix[doc_mask]

            if len(matrix) == 0:
                positive = np.array([0.0])
                neutral = np.array([0.0])
                negative = np.array([0.0])
            else:
                label_columns = {
                    label: matrix[:, index]
                    for index, label in id_to_label.items()
                }
                positive = label_columns.get(
                    "positive",
                    np.zeros(len(matrix)),
                )
                neutral = label_columns.get(
                    "neutral",
                    np.zeros(len(matrix)),
                )
                negative = label_columns.get(
                    "negative",
                    np.zeros(len(matrix)),
                )

            feature_rows.append(
                {
                    "finbert_text_hash": row.finbert_text_hash,
                    "finbert_positive_mean": float(positive.mean()),
                    "finbert_neutral_mean": float(neutral.mean()),
                    "finbert_negative_mean": float(negative.mean()),
                    "finbert_negative_max": float(negative.max()),
                    "finbert_negative_std": float(negative.std()),
                    "finbert_positive_minus_negative": float(
                        positive.mean() - negative.mean()
                    ),
                }
            )

        new_features = pd.DataFrame(feature_rows)
        if cached.empty:
            cached = new_features
        else:
            cached = pd.concat(
                [cached, new_features],
                ignore_index=True,
            ).drop_duplicates(
                "finbert_text_hash",
                keep="last",
            )

        cached.to_parquet(cache_path, index=False)

    output = output.merge(
        cached[["finbert_text_hash"] + feature_names],
        on="finbert_text_hash",
        how="left",
    )

    return output, feature_names


if TEXT_MODELS_READY:
    text_model_data, finbert_feature_columns = (
        build_finbert_sentiment_features(text_model_data)
    )
else:
    finbert_feature_columns = []

print("FinBERT features included:", finbert_feature_columns)

## Filing-language change features

This experiment augments the **level** of current filing language with the
quarter-to-quarter change in that language.

For sentence embeddings:

\[
\Delta E_t = E_t - E_{t-1}
\]

For FinBERT sentiment:

\[
\Delta S_t = S_t - S_{t-1}
\]

These are the textual analogue of financial momentum/change features.

In [ ]:
def add_filing_language_change_features(frame):
    """Add quarter-over-quarter changes for numeric text features.

    Deltas are created BEFORE model fitting in v5 so they can actually be
    selected and evaluated by the main models.
    """
    result = frame.copy()

    company_col = next(
        (c for c in ["cik", "ticker", "company_name"] if c in result.columns),
        None,
    )
    date_col = next(
        (
            c for c in [
                "feature_cutoff_date",
                "source_filing_date",
                "filing_date",
                "quarter_end",
            ]
            if c in result.columns
        ),
        None,
    )
    if company_col is None or date_col is None:
        raise ValueError("Company/date columns are required for language-change features.")

    result[date_col] = pd.to_datetime(result[date_col], errors="coerce")
    result = result.sort_values([company_col, date_col]).copy()

    embedding_level_columns = [
        c for c in result.columns if c.startswith("text_embedding_")
    ]
    known_finbert_features = [
        "finbert_positive_mean",
        "finbert_neutral_mean",
        "finbert_negative_mean",
        "finbert_negative_max",
        "finbert_negative_std",
        "finbert_positive_minus_negative",
    ]
    finbert_level_columns = [
        c for c in known_finbert_features if c in result.columns
    ]

    level_columns = embedding_level_columns + finbert_level_columns
    if not level_columns:
        print("No numeric Sentence Transformer / FinBERT level features found.")
        return result

    for column in level_columns:
        result[column] = pd.to_numeric(result[column], errors="coerce")
        result[f"{column}_delta1"] = (
            result.groupby(company_col, dropna=False)[column].diff()
        )

    print(
        "Language-change source features:",
        len(level_columns),
        f"({len(embedding_level_columns)} embeddings + "
        f"{len(finbert_level_columns)} FinBERT)",
    )
    return result

text_model_data = add_filing_language_change_features(text_model_data)

embedding_delta_columns = [
    c for c in text_model_data.columns
    if c.startswith("text_embedding_") and c.endswith("_delta1")
]
finbert_delta_columns = [
    c for c in text_model_data.columns
    if c.startswith("finbert_") and c.endswith("_delta1")
]

print("Sentence embedding delta features:", len(embedding_delta_columns))
print("FinBERT delta features:", len(finbert_delta_columns))


## Pre-fit data and text audit


In [ ]:
# v5 data / text audit before model fitting
audit_rows = []
for split_name in ["train", "validation", "test"]:
    part = text_model_data[text_model_data["split"] == split_name]
    audit_rows.append({
        "split": split_name,
        "rows": len(part),
        "companies": part["cik"].nunique() if "cik" in part.columns else np.nan,
        "acceleration_rate": float(part["target_clean"].mean()) if len(part) else np.nan,
        "median_text_chars": float(part["sec_text"].astype(str).str.len().median()) if len(part) else np.nan,
    })

data_audit = pd.DataFrame(audit_rows)
display(data_audit)

duplicate_keys = [
    c for c in ["cik", "quarter_end"] if c in text_model_data.columns
]
if len(duplicate_keys) == 2:
    duplicate_count = int(text_model_data.duplicated(duplicate_keys).sum())
    print("Duplicate company-quarter rows:", duplicate_count)

if {"source_filing_date", "feature_cutoff_date"}.issubset(text_model_data.columns):
    source_date = pd.to_datetime(text_model_data["source_filing_date"], errors="coerce")
    cutoff_date = pd.to_datetime(text_model_data["feature_cutoff_date"], errors="coerce")
    future_text_rows = int((source_date > cutoff_date).fillna(False).sum())
    print("Rows where source filing date is AFTER feature cutoff:", future_text_rows)


## Train text-only logistic-regression models

In [ ]:
def choose_threshold(y_true, probabilities):
    """Select the validation threshold that maximizes balanced accuracy.

    Ties are resolved by choosing the threshold closest to 0.50.
    """
    scored = []
    for threshold in THRESHOLD_GRID:
        pred = (probabilities >= threshold).astype(int)
        scored.append((
            float(threshold),
            float(balanced_accuracy_score(y_true, pred)),
        ))
    best_score = max(score for _, score in scored)
    tied = [
        item for item in scored
        if np.isclose(item[1], best_score, rtol=0.0, atol=1e-12)
    ]
    return min(tied, key=lambda item: abs(item[0] - 0.50))


def evaluation_row(name, y_true, p, pred, base_p):
    base = np.full(len(y_true), base_p)
    base_brier = brier_score_loss(y_true, base)
    brier = brier_score_loss(y_true, p)
    return {
        "model": name,
        "rows": len(y_true),
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "macro_f1": f1_score(y_true, pred, average="macro", zero_division=0),
        "acceleration_precision": precision_score(y_true, pred, pos_label=1, zero_division=0),
        "acceleration_recall": recall_score(y_true, pred, pos_label=1, zero_division=0),
        "deceleration_recall": recall_score(y_true, pred, pos_label=0, zero_division=0),
        "brier_score": brier,
        "brier_skill_score": 1 - brier / base_brier if base_brier > 0 else np.nan,
        "roc_auc": roc_auc_score(y_true, p) if pd.Series(y_true).nunique() == 2 else np.nan,
        "average_precision": average_precision_score(y_true, p) if pd.Series(y_true).nunique() == 2 else np.nan,
    }


def is_better_candidate(candidate, best):
    """Primary objective = validation balanced accuracy; Brier only tie-breaks."""
    if best is None:
        return True
    if candidate["val_bal"] > best["val_bal"] + 1e-12:
        return True
    if np.isclose(candidate["val_bal"], best["val_bal"], rtol=0.0, atol=1e-12):
        if candidate["brier"] < best["brier"] - 1e-12:
            return True
    return False


def dense_text_pipeline(
    use_embeddings,
    use_finbert,
    use_deltas,
    C,
    class_weight,
    pca_components,
    train_rows,
):
    branches = []

    if use_embeddings:
        embedding_features = list(text_embedding_columns)
        if use_deltas:
            embedding_features += list(embedding_delta_columns)

        ncomp = max(
            1,
            min(
                int(pca_components),
                len(embedding_features),
                train_rows - 1,
            ),
        )
        branches.append((
            "sentence_embeddings",
            Pipeline([
                ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=ncomp, random_state=RANDOM_STATE)),
            ]),
            embedding_features,
        ))

    if use_finbert:
        fin_features = list(finbert_feature_columns)
        if use_deltas:
            fin_features += list(finbert_delta_columns)

        branches.append((
            "finbert",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            fin_features,
        ))

    prep = ColumnTransformer(
        branches,
        remainder="drop",
        verbose_feature_names_out=False,
    )
    return Pipeline([
        ("preprocessor", prep),
        ("classifier", LogisticRegression(
            C=C,
            penalty="l2",
            class_weight=class_weight,
            solver="lbfgs",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ])


train = text_model_data[text_model_data.split == "train"].copy()
val = text_model_data[text_model_data.split == "validation"].copy()
test = text_model_data[text_model_data.split == "test"].copy()
y_train = train.target_clean.astype(int)
y_val = val.target_clean.astype(int)
y_test = test.target_clean.astype(int)

base_p = float(y_train.mean())
min_df_default = 1 if len(train) < 100 else 2

rows = []
selection = []
models = {}
preds = test[[
    c for c in [
        "cik", "ticker", "company_name", "quarter_end",
        "future_growth_change", "target_clean",
    ]
    if c in test.columns
]].copy()

prior = np.full(len(test), base_p)
rows.append(evaluation_row(
    "Prior-probability baseline",
    y_test,
    prior,
    (prior >= 0.5).astype(int),
    base_p,
))

# -----------------------------
# TF-IDF baseline tuning
# -----------------------------
tfidf_best = None
tfidf_grid = [
    {"ngram_range": (1, 1), "max_features": 10000},
    {"ngram_range": (1, 2), "max_features": 20000},
]

for tfidf_params in tfidf_grid:
    for w in [None, "balanced"]:
        for C in C_GRID:
            pipe = Pipeline([
                ("tfidf", TfidfVectorizer(
                    lowercase=True,
                    strip_accents="unicode",
                    stop_words="english",
                    ngram_range=tfidf_params["ngram_range"],
                    min_df=min_df_default,
                    max_df=0.98,
                    max_features=tfidf_params["max_features"],
                    sublinear_tf=True,
                )),
                ("classifier", LogisticRegression(
                    C=C,
                    penalty="l2",
                    class_weight=w,
                    solver="liblinear",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                )),
            ])
            pipe.fit(train["sec_text"], y_train)
            val_p = pipe.predict_proba(val["sec_text"])[:, 1]
            threshold, val_bal = choose_threshold(y_val, val_p)
            candidate = {
                "C": C,
                "class_weight": w,
                "threshold": float(threshold),
                "val_bal": float(val_bal),
                "brier": float(brier_score_loss(y_val, val_p)),
                **tfidf_params,
            }
            if is_better_candidate(candidate, tfidf_best):
                tfidf_best = candidate

tfidf_final = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        stop_words="english",
        ngram_range=tfidf_best["ngram_range"],
        min_df=min_df_default,
        max_df=0.98,
        max_features=tfidf_best["max_features"],
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(
        C=tfidf_best["C"],
        penalty="l2",
        class_weight=tfidf_best["class_weight"],
        solver="liblinear",
        max_iter=5000,
        random_state=RANDOM_STATE,
    )),
])
tfidf_final.fit(train["sec_text"], y_train)
test_p = tfidf_final.predict_proba(test["sec_text"])[:, 1]
test_pred = (test_p >= tfidf_best["threshold"]).astype(int)

row = evaluation_row(
    "TF-IDF + Logistic Regression",
    y_test, test_p, test_pred, base_p,
)
row.update({
    "selected_C": tfidf_best["C"],
    "class_weight": str(tfidf_best["class_weight"]),
    "threshold": tfidf_best["threshold"],
    "validation_balanced_accuracy": tfidf_best["val_bal"],
    "validation_brier": tfidf_best["brier"],
    "selected_pca_components": np.nan,
    "selected_ngram_range": str(tfidf_best["ngram_range"]),
    "selected_max_features": tfidf_best["max_features"],
})
rows.append(row)
selection.append({"model": "TF-IDF + Logistic Regression", **tfidf_best})
models["TF-IDF + Logistic Regression"] = tfidf_final
preds["TF-IDF + Logistic Regression_probability"] = test_p
preds["TF-IDF + Logistic Regression_prediction"] = test_pred

# -----------------------------
# Dense text model tuning
# -----------------------------
dense_specs = [
    ("Sentence Transformer + Logistic Regression", True, False, False),
    ("Sentence Transformer + Delta + Logistic Regression", True, False, True),
    ("FinBERT + Logistic Regression", False, True, False),
    ("FinBERT + Delta + Logistic Regression", False, True, True),
    ("Sentence Transformer + FinBERT + Logistic Regression", True, True, False),
    ("Sentence Transformer + FinBERT + Delta + Logistic Regression", True, True, True),
]

for name, use_embeddings, use_finbert, use_deltas in dense_specs:
    best = None
    pca_grid = TEXT_PCA_GRID if use_embeddings else [1]

    for pca_components in pca_grid:
        for w in [None, "balanced"]:
            for C in C_GRID:
                pipe = dense_text_pipeline(
                    use_embeddings,
                    use_finbert,
                    use_deltas,
                    C,
                    w,
                    pca_components,
                    len(train),
                )
                pipe.fit(train, y_train)
                val_p = pipe.predict_proba(val)[:, 1]
                threshold, val_bal = choose_threshold(y_val, val_p)
                candidate = {
                    "C": C,
                    "class_weight": w,
                    "threshold": float(threshold),
                    "val_bal": float(val_bal),
                    "brier": float(brier_score_loss(y_val, val_p)),
                    "pca_components": int(pca_components) if use_embeddings else np.nan,
                    "use_deltas": bool(use_deltas),
                }
                if is_better_candidate(candidate, best):
                    best = candidate

    final = dense_text_pipeline(
        use_embeddings,
        use_finbert,
        use_deltas,
        best["C"],
        best["class_weight"],
        best["pca_components"] if use_embeddings else 1,
        len(train),
    )
    # No refit on train+validation: threshold was calibrated from a train-fitted model.
    final.fit(train, y_train)

    test_p = final.predict_proba(test)[:, 1]
    test_pred = (test_p >= best["threshold"]).astype(int)
    row = evaluation_row(name, y_test, test_p, test_pred, base_p)
    row.update({
        "selected_C": best["C"],
        "class_weight": str(best["class_weight"]),
        "threshold": best["threshold"],
        "validation_balanced_accuracy": best["val_bal"],
        "validation_brier": best["brier"],
        "selected_pca_components": best["pca_components"],
        "uses_language_deltas": best["use_deltas"],
    })
    rows.append(row)
    selection.append({"model": name, **best})
    models[name] = final
    preds[f"{name}_probability"] = test_p
    preds[f"{name}_prediction"] = test_pred

results = pd.DataFrame(rows).sort_values(
    ["balanced_accuracy", "brier_score"],
    ascending=[False, True],
)
selection = pd.DataFrame(selection).sort_values(
    ["val_bal", "brier"],
    ascending=[False, True],
)

display(results)
display(selection)

results.to_csv(OUTPUT_DIR / "textual_test_results.csv", index=False)
selection.to_csv(OUTPUT_DIR / "textual_validation_selection.csv", index=False)
preds.to_csv(OUTPUT_DIR / "textual_test_predictions.csv", index=False)

with pd.ExcelWriter(
    OUTPUT_DIR / "textual_branch_results.xlsx",
    engine="openpyxl",
) as writer:
    results.to_excel(writer, sheet_name="Model_Results", index=False)
    selection.to_excel(writer, sheet_name="Validation_Selection", index=False)
    preds.to_excel(writer, sheet_name="Test_Predictions", index=False)
    data_audit.to_excel(writer, sheet_name="Data_Audit", index=False)

joblib.dump(models, OUTPUT_DIR / "textual_final_models.joblib")
print("Saved to:", OUTPUT_DIR)


## v6 — Stronger model families for dense text features

Random Forest and XGBoost are now tested on the same Sentence Transformer / FinBERT representations as the Logistic Regression baselines. TF-IDF stays with Logistic Regression because it is a high-dimensional sparse representation.


In [ ]:

# ================================================================
# v6 — RANDOM FOREST + XGBOOST TEXT-ONLY EXPERIMENTS
# ================================================================

# These are the same Random Forest / XGBoost candidate families used in
# Branch 1 numerical modeling. We tune them again here because the optimal
# settings for dense text features need not be identical to the optimal
# settings for accounting features.
RF_PARAM_GRID = [
    {
        "n_estimators": 300,
        "max_depth": None,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 500,
        "max_depth": 6,
        "min_samples_leaf": 2,
        "max_features": "sqrt",
    },
    {
        "n_estimators": 500,
        "max_depth": 10,
        "min_samples_leaf": 3,
        "max_features": 0.7,
    },
]

XGB_PARAM_GRID = [
    {
        "n_estimators": 200,
        "max_depth": 2,
        "learning_rate": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
    },
    {
        "n_estimators": 300,
        "max_depth": 3,
        "learning_rate": 0.03,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
    },
    {
        "n_estimators": 300,
        "max_depth": 4,
        "learning_rate": 0.02,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
    },
]

# Smaller tree-specific PCA grid keeps this experiment practical while still
# testing whether more/less semantic compression helps.
TREE_TEXT_PCA_GRID = [16, 32, 64]


def make_dense_text_preprocessor(
    mode,
    train_rows,
    pca_components=32,
):
    """Compact dense text representation for RF/XGBoost."""
    branches = []

    use_sentence = mode in {
        "sentence",
        "sentence_delta",
        "sentence_finbert",
        "sentence_finbert_delta",
    }
    use_finbert = mode in {
        "finbert",
        "finbert_delta",
        "sentence_finbert",
        "sentence_finbert_delta",
    }
    use_delta = mode.endswith("_delta")

    if use_sentence:
        sentence_features = list(text_embedding_columns)
        if use_delta:
            sentence_features += list(embedding_delta_columns)

        n_components = max(
            1,
            min(
                int(pca_components),
                len(sentence_features),
                train_rows - 1,
            ),
        )

        branches.append((
            "sentence",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(
                        strategy="constant",
                        fill_value=0.0,
                    ),
                ),
                ("scaler", StandardScaler()),
                (
                    "pca",
                    PCA(
                        n_components=n_components,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]),
            sentence_features,
        ))

    if use_finbert:
        finbert_features = list(finbert_feature_columns)
        if use_delta:
            finbert_features += list(finbert_delta_columns)

        branches.append((
            "finbert",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median"),
                ),
            ]),
            finbert_features,
        ))

    if not branches:
        raise ValueError(
            f"No dense text branches were selected for mode={mode}"
        )

    return ColumnTransformer(
        branches,
        remainder="drop",
        verbose_feature_names_out=False,
    )


def build_random_forest_text_model(
    mode,
    params,
    train_rows,
    pca_components=32,
):
    return Pipeline([
        (
            "preprocessor",
            make_dense_text_preprocessor(
                mode,
                train_rows,
                pca_components,
            ),
        ),
        (
            "classifier",
            RandomForestClassifier(
                **params,
                class_weight="balanced_subsample",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ])


def xgb_compute_kwargs():
    """Use Kaggle CUDA when the installed XGBoost version supports it."""
    try:
        import torch
        if torch.cuda.is_available():
            return {
                "tree_method": "hist",
                "device": "cuda",
            }
    except Exception:
        pass

    return {
        "tree_method": "hist",
    }


def build_xgboost_text_model(
    mode,
    params,
    train_rows,
    pca_components=32,
):
    positive = max(1, int((y_train == 1).sum()))
    negative = max(1, int((y_train == 0).sum()))

    return Pipeline([
        (
            "preprocessor",
            make_dense_text_preprocessor(
                mode,
                train_rows,
                pca_components,
            ),
        ),
        (
            "classifier",
            XGBClassifier(
                **params,
                scale_pos_weight=negative / positive,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                **xgb_compute_kwargs(),
            ),
        ),
    ])


TEXT_TREE_MODES = [
    (
        "Sentence Transformer",
        "sentence",
    ),
    (
        "Sentence Transformer + Delta",
        "sentence_delta",
    ),
    (
        "FinBERT",
        "finbert",
    ),
    (
        "FinBERT + Delta",
        "finbert_delta",
    ),
    (
        "Sentence Transformer + FinBERT",
        "sentence_finbert",
    ),
    (
        "Sentence Transformer + FinBERT + Delta",
        "sentence_finbert_delta",
    ),
]


def tune_tree_text_family(
    family_name,
    parameter_grid,
):
    family_rows = []
    family_selection = []
    family_models = {}

    for label, mode in TEXT_TREE_MODES:
        uses_sentence = "sentence" in mode

        pca_grid = (
            TREE_TEXT_PCA_GRID
            if uses_sentence
            else [1]
        )

        total_candidates = (
            len(parameter_grid) * len(pca_grid)
        )

        print("\n" + "=" * 80)
        print(
            f"{family_name}: {label} "
            f"({total_candidates} validation candidates)"
        )
        print("=" * 80)

        best = None
        candidate_number = 0

        for params in parameter_grid:
            for pca_components in pca_grid:
                candidate_number += 1

                if family_name == "Random Forest":
                    model = build_random_forest_text_model(
                        mode,
                        params,
                        len(train),
                        pca_components,
                    )
                else:
                    model = build_xgboost_text_model(
                        mode,
                        params,
                        len(train),
                        pca_components,
                    )

                print(
                    f"[{candidate_number}/{total_candidates}]",
                    "PCA=",
                    (
                        pca_components
                        if uses_sentence
                        else "N/A"
                    ),
                    "params=",
                    params,
                )

                model.fit(train, y_train)

                val_probability = model.predict_proba(val)[:, 1]

                threshold, val_bal = choose_threshold(
                    y_val,
                    val_probability,
                )

                candidate = {
                    "params": params,
                    "threshold": float(threshold),
                    "val_bal": float(val_bal),
                    "brier": float(
                        brier_score_loss(
                            y_val,
                            val_probability,
                        )
                    ),
                    "pca_components": (
                        int(pca_components)
                        if uses_sentence
                        else np.nan
                    ),
                }

                if is_better_candidate(
                    candidate,
                    best,
                ):
                    best = candidate

        # The selected threshold came from a TRAIN-fitted model,
        # therefore fit the test model on TRAIN only as well.
        if family_name == "Random Forest":
            final_model = build_random_forest_text_model(
                mode,
                best["params"],
                len(train),
                (
                    best["pca_components"]
                    if uses_sentence
                    else 1
                ),
            )
        else:
            final_model = build_xgboost_text_model(
                mode,
                best["params"],
                len(train),
                (
                    best["pca_components"]
                    if uses_sentence
                    else 1
                ),
            )

        final_model.fit(train, y_train)

        test_probability = final_model.predict_proba(
            test
        )[:, 1]

        test_prediction = (
            test_probability >= best["threshold"]
        ).astype(int)

        model_name = (
            f"{label} + {family_name}"
        )

        result_row = evaluation_row(
            model_name,
            y_test,
            test_probability,
            test_prediction,
            base_p,
        )

        result_row.update({
            "model_family": family_name,
            "selected_params": str(
                best["params"]
            ),
            "selected_threshold": (
                best["threshold"]
            ),
            "validation_balanced_accuracy": (
                best["val_bal"]
            ),
            "validation_brier": best["brier"],
            "selected_pca_components": (
                best["pca_components"]
            ),
        })

        family_rows.append(result_row)

        family_selection.append({
            "model": model_name,
            **best,
        })

        family_models[model_name] = final_model

        preds[
            f"{model_name}_probability"
        ] = test_probability

        preds[
            f"{model_name}_prediction"
        ] = test_prediction

        print(
            "Selected:",
            model_name,
            "| validation BA =",
            round(best["val_bal"], 4),
            "| test BA =",
            round(
                result_row[
                    "balanced_accuracy"
                ],
                4,
            ),
        )

    return (
        pd.DataFrame(family_rows),
        pd.DataFrame(family_selection),
        family_models,
    )


rf_text_results, rf_text_selection, rf_text_models = (
    tune_tree_text_family(
        "Random Forest",
        RF_PARAM_GRID,
    )
)

xgb_text_results, xgb_text_selection, xgb_text_models = (
    tune_tree_text_family(
        "XGBoost",
        XGB_PARAM_GRID,
    )
)

# Existing v5 `results` contains the Logistic Regression controls.
all_textual_results = pd.concat(
    [
        results.assign(
            experiment_family="Logistic Regression"
        ),
        rf_text_results.assign(
            experiment_family="Random Forest"
        ),
        xgb_text_results.assign(
            experiment_family="XGBoost"
        ),
    ],
    ignore_index=True,
    sort=False,
)

all_textual_results = (
    all_textual_results
    .sort_values(
        [
            "balanced_accuracy",
            "brier_score",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

tree_text_selection = pd.concat(
    [
        rf_text_selection.assign(
            model_family="Random Forest"
        ),
        xgb_text_selection.assign(
            model_family="XGBoost"
        ),
    ],
    ignore_index=True,
    sort=False,
)

models.update(rf_text_models)
models.update(xgb_text_models)

print("\n" + "=" * 80)
print("FINAL v6 TEXTUAL MODEL RANKING")
print("=" * 80)

display(all_textual_results)
display(tree_text_selection)

all_textual_results.to_csv(
    OUTPUT_DIR / "all_textual_model_results_v6.csv",
    index=False,
)

tree_text_selection.to_csv(
    OUTPUT_DIR / "tree_text_validation_selection_v6.csv",
    index=False,
)

preds.to_csv(
    OUTPUT_DIR / "textual_test_predictions_v6.csv",
    index=False,
)

with pd.ExcelWriter(
    OUTPUT_DIR / "textual_branch_v6_results.xlsx",
    engine="openpyxl",
) as writer:
    all_textual_results.to_excel(
        writer,
        sheet_name="All_Model_Results",
        index=False,
    )

    tree_text_selection.to_excel(
        writer,
        sheet_name="Tree_Validation_Selection",
        index=False,
    )

    selection.to_excel(
        writer,
        sheet_name="LR_Validation_Selection",
        index=False,
    )

    data_audit.to_excel(
        writer,
        sheet_name="Data_Audit",
        index=False,
    )

joblib.dump(
    models,
    OUTPUT_DIR / "textual_v6_final_models.joblib",
)

print("Saved v6 textual results to:", OUTPUT_DIR)


## v7 — Filing-section and shuffled-text ablation

This keeps rows/splits fixed while changing only which SEC text is supplied. If MD&A beats MD&A + Risk Factors, the added Risk Factors section may be acting as noise. If real text performs like shuffled text, the representation contains little usable predictive signal.


In [ ]:
# ================================================================
# v7 TEXT-NOISE ABLATION
# ================================================================
SECTION_MIN_CHARS = 300
ABLATION_RANDOM_SEED = 2027

MDA_COLUMNS = [
    c for c in ["mda_text", "management_discussion_text"]
    if c in text_model_data.columns
]
RISK_COLUMNS = [
    c for c in ["risk_factors_text", "risk_text"]
    if c in text_model_data.columns
]


def _combine_columns(frame, columns):
    if not columns:
        return pd.Series("", index=frame.index, dtype="object")
    return (
        frame[columns].fillna("").astype(str).apply(
            lambda row: "\n\n".join(
                value.strip() for value in row
                if value and value.strip()
            ), axis=1
        )
    )


ablation_frame = text_model_data.copy()
ablation_frame["text_mda"] = _combine_columns(ablation_frame, MDA_COLUMNS)
ablation_frame["text_risk"] = _combine_columns(ablation_frame, RISK_COLUMNS)
ablation_frame["text_mda_risk"] = (
    ablation_frame["text_mda"].fillna("") + "\n\n" +
    ablation_frame["text_risk"].fillna("")
).str.strip()
ablation_frame["text_full"] = ablation_frame["sec_text"].fillna("").astype(str)

print("Detected MD&A columns:", MDA_COLUMNS)
print("Detected Risk Factors columns:", RISK_COLUMNS)

if not MDA_COLUMNS:
    print("WARNING: no dedicated MD&A column found; MD&A-only ablation will be skipped.")
if not RISK_COLUMNS:
    print("WARNING: no dedicated Risk Factors column found; risk-only ablation will be skipped.")

# SAME ROWS for every section comparison.  When dedicated sections exist,
# require both to be present so performance changes cannot be explained by
# evaluating on different observations.
mask = (
    ablation_frame["target_clean"].notna() &
    ablation_frame["split"].isin(["train", "validation", "test"])
)
if MDA_COLUMNS:
    mask &= ablation_frame["text_mda"].str.len().ge(SECTION_MIN_CHARS)
if RISK_COLUMNS:
    mask &= ablation_frame["text_risk"].str.len().ge(SECTION_MIN_CHARS)

section_data = ablation_frame.loc[mask].copy()
if section_data.empty:
    raise ValueError("No common rows remain for text-section ablation.")

if "observation_id" not in section_data.columns:
    q = pd.to_datetime(section_data["quarter_end"], errors="coerce").dt.strftime("%Y-%m-%d")
    company = (section_data["cik"].astype(str) if "cik" in section_data.columns
               else section_data["ticker"].astype(str))
    section_data["observation_id"] = company.str.strip() + "__" + q.fillna("missing_date")

section_manifest = section_data[[c for c in [
    "observation_id", "split", "cik", "ticker", "company_name",
    "quarter_end", "target_clean"
] if c in section_data.columns]].sort_values(["split", "observation_id"]).reset_index(drop=True)

print("\nSection-ablation rows:")
print(section_manifest.groupby("split").size().rename("rows"))

# Negative control: shuffle full text only WITHIN split, preserving row counts
# and the marginal text distribution without preserving the text-label link.
rng = np.random.default_rng(ABLATION_RANDOM_SEED)
section_data["text_full_shuffled"] = ""
for split_name in ["train", "validation", "test"]:
    idx = section_data.index[section_data["split"].eq(split_name)].to_numpy()
    values = section_data.loc[idx, "text_full"].to_numpy(copy=True)
    rng.shuffle(values)
    section_data.loc[idx, "text_full_shuffled"] = values

TEXT_VARIANTS = {
    "Full available filing text": "text_full",
    "Shuffled full text": "text_full_shuffled",
}
if MDA_COLUMNS:
    TEXT_VARIANTS["MD&A only"] = "text_mda"
if RISK_COLUMNS:
    TEXT_VARIANTS["Risk Factors only"] = "text_risk"
if MDA_COLUMNS and RISK_COLUMNS:
    TEXT_VARIANTS["MD&A + Risk Factors"] = "text_mda_risk"

train_a = section_data[section_data["split"].eq("train")].copy()
val_a = section_data[section_data["split"].eq("validation")].copy()
test_a = section_data[section_data["split"].eq("test")].copy()
y_train_a = train_a["target_clean"].astype(int)
y_val_a = val_a["target_clean"].astype(int)
y_test_a = test_a["target_clean"].astype(int)
base_p_a = float(y_train_a.mean())
min_df_a = 1 if len(train_a) < 100 else 2


def _tune_tfidf(text_col):
    best = None
    for ngram_range, max_features in [((1, 1), 10000), ((1, 2), 20000)]:
        for class_weight in [None, "balanced"]:
            for C in C_GRID:
                model = Pipeline([
                    ("tfidf", TfidfVectorizer(
                        lowercase=True, strip_accents="unicode",
                        stop_words="english", ngram_range=ngram_range,
                        min_df=min_df_a, max_df=0.98,
                        max_features=max_features, sublinear_tf=True,
                    )),
                    ("classifier", LogisticRegression(
                        C=C, penalty="l2", class_weight=class_weight,
                        solver="liblinear", max_iter=5000,
                        random_state=RANDOM_STATE,
                    )),
                ])
                model.fit(train_a[text_col], y_train_a)
                p = model.predict_proba(val_a[text_col])[:, 1]
                threshold, val_bal = choose_threshold(y_val_a, p)
                cand = {
                    "C": C, "class_weight": class_weight,
                    "threshold": float(threshold), "val_bal": float(val_bal),
                    "brier": float(brier_score_loss(y_val_a, p)),
                    "ngram_range": ngram_range, "max_features": max_features,
                }
                if is_better_candidate(cand, best):
                    best = cand
    final = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, strip_accents="unicode", stop_words="english",
            ngram_range=best["ngram_range"], min_df=min_df_a, max_df=0.98,
            max_features=best["max_features"], sublinear_tf=True,
        )),
        ("classifier", LogisticRegression(
            C=best["C"], penalty="l2", class_weight=best["class_weight"],
            solver="liblinear", max_iter=5000, random_state=RANDOM_STATE,
        )),
    ])
    final.fit(train_a[text_col], y_train_a)
    p = final.predict_proba(test_a[text_col])[:, 1]
    pred = (p >= best["threshold"]).astype(int)
    return best, p, pred


rows = []
for variant_name, text_col in TEXT_VARIANTS.items():
    print("\nRunning text-noise ablation:", variant_name)
    best, p, pred = _tune_tfidf(text_col)
    row = evaluation_row(
        f"{variant_name} | TF-IDF + LR",
        y_test_a, p, pred, base_p_a,
    )
    row.update({
        "text_variant": variant_name,
        "model_family": "TF-IDF + Logistic Regression",
        "selected_C": best["C"],
        "selected_class_weight": str(best["class_weight"]),
        "selected_threshold": best["threshold"],
        "validation_balanced_accuracy": best["val_bal"],
        "validation_brier": best["brier"],
    })
    rows.append(row)

text_noise_ablation_results = pd.DataFrame(rows)

# Difference from the full-text representation. Positive means that removing
# some text improved balanced accuracy, which is direct evidence that the
# removed material may be acting as noise for this representation.
full_ba = float(
    text_noise_ablation_results.loc[
        text_noise_ablation_results["text_variant"].eq("Full available filing text"),
        "balanced_accuracy",
    ].iloc[0]
)
text_noise_ablation_results["balanced_accuracy_delta_vs_full_text"] = (
    text_noise_ablation_results["balanced_accuracy"] - full_ba
)

# Real-text minus shuffled-text control.
shuffle_ba = float(
    text_noise_ablation_results.loc[
        text_noise_ablation_results["text_variant"].eq("Shuffled full text"),
        "balanced_accuracy",
    ].iloc[0]
)
text_noise_ablation_results["full_text_minus_shuffled_ba"] = np.where(
    text_noise_ablation_results["text_variant"].eq("Full available filing text"),
    full_ba - shuffle_ba,
    np.nan,
)

text_noise_ablation_results = text_noise_ablation_results.sort_values(
    "balanced_accuracy", ascending=False
).reset_index(drop=True)

display(text_noise_ablation_results[[c for c in [
    "text_variant", "rows", "accuracy", "balanced_accuracy",
    "balanced_accuracy_delta_vs_full_text", "macro_f1", "roc_auc",
    "brier_score", "brier_skill_score", "validation_balanced_accuracy"
] if c in text_noise_ablation_results.columns]])

print("\nINTERPRETATION CHECKS")
print("Full text BA:", round(full_ba, 4))
print("Shuffled full text BA:", round(shuffle_ba, 4))
print("Full - shuffled BA:", round(full_ba - shuffle_ba, 4))

if MDA_COLUMNS:
    mda_ba = float(text_noise_ablation_results.loc[
        text_noise_ablation_results["text_variant"].eq("MD&A only"),
        "balanced_accuracy"
    ].iloc[0])
    print("MD&A only BA:", round(mda_ba, 4))
    print("MD&A only - full BA:", round(mda_ba - full_ba, 4))

if MDA_COLUMNS and RISK_COLUMNS:
    mda_ba = float(text_noise_ablation_results.loc[
        text_noise_ablation_results["text_variant"].eq("MD&A only"),
        "balanced_accuracy"
    ].iloc[0])
    mda_risk_ba = float(text_noise_ablation_results.loc[
        text_noise_ablation_results["text_variant"].eq("MD&A + Risk Factors"),
        "balanced_accuracy"
    ].iloc[0])
    print("(MD&A + Risk) - MD&A BA:", round(mda_risk_ba - mda_ba, 4))
    if mda_risk_ba < mda_ba:
        print("Risk Factors lowered BA when added to MD&A: evidence of added noise.")
    elif mda_risk_ba > mda_ba:
        print("Risk Factors improved BA when added to MD&A: evidence of incremental signal.")
    else:
        print("Risk Factors produced no measurable BA change.")

section_manifest.to_csv(
    OUTPUT_DIR / "text_section_ablation_equal_rows.csv", index=False
)
text_noise_ablation_results.to_csv(
    OUTPUT_DIR / "text_noise_ablation_results_v7.csv", index=False
)
with pd.ExcelWriter(
    OUTPUT_DIR / "text_noise_ablation_v7.xlsx", engine="openpyxl"
) as writer:
    text_noise_ablation_results.to_excel(writer, sheet_name="Ablation_Results", index=False)
    section_manifest.to_excel(writer, sheet_name="Equal_Row_Manifest", index=False)

print("Saved text-noise ablation outputs to:", OUTPUT_DIR)


## v8.1 strong-family section ablation + repeated shuffle control

This corrected version includes the missing section-specific Sentence Transformer
embedding helper and uses the existing TF-IDF ablation helper names consistently.


In [ ]:
best_dense_selection = (
    tree_text_selection
    .sort_values(["val_bal","brier"], ascending=[False,True])
    .iloc[0]
)
BEST_DENSE_FAMILY = best_dense_selection["model_family"]
print("Validation-selected dense family:", BEST_DENSE_FAMILY)

def build_dense_section_estimator(family_name, params, pca_n, y_local):
    if family_name == "Random Forest":
        clf = RandomForestClassifier(
            **params, class_weight="balanced_subsample",
            random_state=RANDOM_STATE, n_jobs=-1,
        )
    elif family_name == "XGBoost":
        pos = max(1, int((y_local == 1).sum()))
        neg = max(1, int((y_local == 0).sum()))
        clf = XGBClassifier(
            **params, scale_pos_weight=neg/pos,
            objective="binary:logistic", eval_metric="logloss",
            random_state=RANDOM_STATE, n_jobs=-1, **xgb_compute_kwargs(),
        )
    else:
        raise ValueError(family_name)

    return Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value=0.0)),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=pca_n, random_state=RANDOM_STATE)),
        ("classifier", clf),
    ])


def build_variant_sentence_embeddings(
    frame: pd.DataFrame,
    text_column: str,
    variant_name: str,
):
    """
    Build one mean-pooled Sentence Transformer embedding per observation
    for a specific filing-text variant (MD&A, Risk Factors, etc.).

    Uses the same chunk_document() logic as the main textual model and caches
    results by text hash.
    """
    from sentence_transformers import SentenceTransformer

    required = {"observation_id", text_column}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(
            f"Section embedding input is missing columns: {sorted(missing)}"
        )

    safe_name = re.sub(
        r"[^A-Za-z0-9_.-]+",
        "_",
        str(variant_name),
    )

    cache_path = (
        TEXT_CACHE_DIR
        / f"section_ablation_{safe_name}_sentence_embeddings.parquet"
    )

    work = frame[
        ["observation_id", text_column]
    ].copy()

    work[text_column] = (
        work[text_column]
        .fillna("")
        .astype(str)
    )

    def _section_hash(text):
        payload = (
            TEXT_EMBEDDING_MODEL
            + "\n"
            + str(CHUNK_WORDS)
            + "\n"
            + str(CHUNK_OVERLAP_WORDS)
            + "\n"
            + str(MAX_CHUNKS_PER_FILING)
            + "\n"
            + str(text)
        )
        return hashlib.sha1(
            payload.encode("utf-8", errors="ignore")
        ).hexdigest()

    work["section_text_hash"] = work[text_column].map(
        _section_hash
    )

    if cache_path.exists():
        cached = pd.read_parquet(cache_path)
    else:
        cached = pd.DataFrame()

    cached_hashes = (
        set(cached["section_text_hash"])
        if (
            not cached.empty
            and "section_text_hash" in cached.columns
        )
        else set()
    )

    unique_docs = (
        work[
            ["section_text_hash", text_column]
        ]
        .drop_duplicates("section_text_hash")
    )

    missing_docs = unique_docs[
        ~unique_docs["section_text_hash"].isin(
            cached_hashes
        )
    ].copy()

    if not missing_docs.empty:
        devices = get_torch_devices()
        print(
            f"{variant_name} Sentence Transformer device(s):",
            devices,
        )

        encoder = SentenceTransformer(
            TEXT_EMBEDDING_MODEL
        )

        flat_chunks = []
        owners = []

        for _, row in missing_docs.iterrows():
            chunks = chunk_document(
                row[text_column]
            )
            if not chunks:
                continue

            flat_chunks.extend(chunks)
            owners.extend(
                [row["section_text_hash"]] * len(chunks)
            )

        if not flat_chunks:
            raise ValueError(
                f"No valid Sentence Transformer chunks for {variant_name}."
            )

        encode_device = (
            devices
            if len(devices) > 1
            else devices[0]
        )

        try:
            chunk_embeddings = encoder.encode(
                flat_chunks,
                batch_size=TEXT_BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
                device=encode_device,
            )
        except Exception as exc:
            if len(devices) <= 1:
                raise

            print(
                "Multi-GPU Sentence Transformer failed; "
                "falling back to the first GPU:",
                repr(exc),
            )

            encoder = SentenceTransformer(
                TEXT_EMBEDDING_MODEL,
                device=devices[0],
            )

            chunk_embeddings = encoder.encode(
                flat_chunks,
                batch_size=TEXT_BATCH_SIZE,
                show_progress_bar=True,
                convert_to_numpy=True,
                normalize_embeddings=True,
                device=devices[0],
            )

        chunk_frame = pd.DataFrame(
            chunk_embeddings
        )
        chunk_frame.insert(
            0,
            "section_text_hash",
            owners,
        )

        new_docs = (
            chunk_frame
            .groupby(
                "section_text_hash",
                sort=False,
            )
            .mean()
            .reset_index()
        )

        raw_embedding_cols = [
            c for c in new_docs.columns
            if c != "section_text_hash"
        ]

        matrix = new_docs[
            raw_embedding_cols
        ].to_numpy()

        norms = np.linalg.norm(
            matrix,
            axis=1,
            keepdims=True,
        )

        matrix = matrix / np.maximum(
            norms,
            1e-12,
        )

        new_docs[
            raw_embedding_cols
        ] = matrix

        if cached.empty:
            cached = new_docs
        else:
            cached = (
                pd.concat(
                    [cached, new_docs],
                    ignore_index=True,
                )
                .drop_duplicates(
                    "section_text_hash",
                    keep="last",
                )
            )

        cached.to_parquet(
            cache_path,
            index=False,
        )

    raw_embedding_cols = [
        c for c in cached.columns
        if c != "section_text_hash"
    ]

    rename_map = {
        c: f"ablation_embedding_{int(c):03d}"
        for c in raw_embedding_cols
    }

    cached_for_merge = cached.rename(
        columns=rename_map
    )

    embedding_columns = list(
        rename_map.values()
    )

    merged = work[
        ["observation_id", "section_text_hash"]
    ].merge(
        cached_for_merge,
        on="section_text_hash",
        how="left",
    )

    if merged[embedding_columns].isna().all(axis=1).any():
        missing_count = int(
            merged[embedding_columns]
            .isna()
            .all(axis=1)
            .sum()
        )
        raise ValueError(
            f"{variant_name}: {missing_count} observations have no embedding."
        )

    return (
        merged[
            ["observation_id"] + embedding_columns
        ],
        embedding_columns,
    )



def run_strong_section_variant(text_column, variant_name):
    emb_frame, emb_cols = build_variant_sentence_embeddings(
        section_data, text_column, variant_name
    )
    merged = section_data.merge(emb_frame, on="observation_id", how="left")
    tr = merged[merged["split"].eq("train")].copy()
    va = merged[merged["split"].eq("validation")].copy()
    te = merged[merged["split"].eq("test")].copy()
    yt = tr["target_clean"].astype(int)
    yv = va["target_clean"].astype(int)
    yte = te["target_clean"].astype(int)

    grid = RF_PARAM_GRID if BEST_DENSE_FAMILY=="Random Forest" else XGB_PARAM_GRID
    best = None
    for params in grid:
        for requested in [16,32,64]:
            pca_n = max(1, min(requested, len(emb_cols), len(tr)-1))
            model = build_dense_section_estimator(
                BEST_DENSE_FAMILY, params, pca_n, yt
            )
            model.fit(tr[emb_cols], yt)
            val_p = model.predict_proba(va[emb_cols])[:,1]
            threshold, val_ba = choose_threshold(yv, val_p)
            cand = {
                "params":params, "pca_components":pca_n,
                "threshold":float(threshold),
                "val_bal":float(val_ba),
                "brier":float(brier_score_loss(yv,val_p)),
            }
            if is_better_candidate(cand,best):
                best = cand

    model = build_dense_section_estimator(
        BEST_DENSE_FAMILY,best["params"],best["pca_components"],yt
    )
    model.fit(tr[emb_cols],yt)
    p = model.predict_proba(te[emb_cols])[:,1]
    pred = (p >= best["threshold"]).astype(int)
    row = evaluation_row(
        f"{variant_name} | Sentence + {BEST_DENSE_FAMILY}",
        yte,p,pred,float(yt.mean())
    )
    row.update({
        "text_variant":variant_name,
        "model_family":BEST_DENSE_FAMILY,
        "validation_balanced_accuracy":best["val_bal"],
        "validation_brier":best["brier"],
        "selected_threshold":best["threshold"],
        "selected_params":str(best["params"]),
        "selected_pca_components":best["pca_components"],
    })
    return row

strong_section_rows = []
for variant_name,text_column in TEXT_VARIANTS.items():
    if variant_name == "Shuffled full text":
        continue
    strong_section_rows.append(
        run_strong_section_variant(text_column,variant_name)
    )
strong_section_ablation_results = pd.DataFrame(strong_section_rows)
display(strong_section_ablation_results.sort_values(
    "balanced_accuracy",ascending=False
))

N_SHUFFLE_REPEATS = 100
real_best, real_p, real_pred = _tune_tfidf("text_full")
real_ba = balanced_accuracy_score(y_test_a, real_pred)
shuffle_ba = []

for repeat in range(N_SHUFFLE_REPEATS):
    shuffled = section_data.copy()
    rng = np.random.default_rng(ABLATION_RANDOM_SEED + 1000 + repeat)

    for split_name in ["train","validation","test"]:
        mask = shuffled["split"].eq(split_name)
        vals = shuffled.loc[mask,"text_full"].to_numpy(copy=True)
        rng.shuffle(vals)
        shuffled.loc[mask,"shuffle_repeat_text"] = vals

    tr = shuffled[shuffled["split"].eq("train")]
    va = shuffled[shuffled["split"].eq("validation")]
    te = shuffled[shuffled["split"].eq("test")]

    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            lowercase=True, strip_accents="unicode", stop_words="english",
            ngram_range=real_best["ngram_range"], min_df=min_df_a,
            max_df=0.98, max_features=real_best["max_features"],
            sublinear_tf=True,
        )),
        ("classifier", LogisticRegression(
            C=real_best["C"], penalty="l2",
            class_weight=real_best["class_weight"],
            solver="liblinear", max_iter=5000, random_state=RANDOM_STATE,
        )),
    ])
    model.fit(tr["shuffle_repeat_text"],tr["target_clean"].astype(int))
    val_p = model.predict_proba(va["shuffle_repeat_text"])[:,1]
    threshold,_ = choose_threshold(va["target_clean"].astype(int),val_p)
    p = model.predict_proba(te["shuffle_repeat_text"])[:,1]
    pred = (p>=threshold).astype(int)
    shuffle_ba.append(
        balanced_accuracy_score(te["target_clean"].astype(int),pred)
    )

shuffle_control_summary = pd.DataFrame([{
    "real_full_text_balanced_accuracy":real_ba,
    "shuffle_repeats":N_SHUFFLE_REPEATS,
    "mean_shuffled_balanced_accuracy":float(np.mean(shuffle_ba)),
    "std_shuffled_balanced_accuracy":float(np.std(shuffle_ba,ddof=1)),
    "shuffled_2p5":float(np.quantile(shuffle_ba,0.025)),
    "shuffled_97p5":float(np.quantile(shuffle_ba,0.975)),
    "fraction_shuffles_ge_real":float(np.mean(np.asarray(shuffle_ba)>=real_ba)),
}])
display(shuffle_control_summary)

strong_section_ablation_results.to_csv(
    OUTPUT_DIR/"strong_family_section_ablation_v8.csv",index=False
)
shuffle_control_summary.to_csv(
    OUTPUT_DIR/"repeated_shuffle_control_v8.csv",index=False
)


## Formal text ablation study

Use the identical chronological rows to compare:
TF-IDF, Sentence Transformer, FinBERT, and Sentence Transformer + FinBERT.
The language-change columns created above are also reported so a level-only
versus level+change experiment can be run without changing the target split.

In [ ]:
# v6 paper-facing textual ablation / model comparison
source_results = (
    all_textual_results
    if "all_textual_results" in globals()
    else results
)

ablation_columns = [
    c for c in [
        "model",
        "experiment_family",
        "model_family",
        "rows",
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "acceleration_recall",
        "deceleration_recall",
        "roc_auc",
        "average_precision",
        "brier_score",
        "brier_skill_score",
        "selected_C",
        "selected_params",
        "threshold",
        "selected_threshold",
        "selected_pca_components",
        "uses_language_deltas",
    ]
    if c in source_results.columns
]

textual_ablation_results = source_results[
    ablation_columns
].copy()

display(textual_ablation_results)


## Robustness — unseen-company generalization

This secondary test reuses already-generated Sentence Transformer / FinBERT
features and tests them on a company that was never present in training.

In [ ]:
if not RUN_LOCO:
    print("LOCO skipped (set RUN_LOCO=True for the paper robustness run).")
else:
    company_col = next(
        (c for c in ["cik", "ticker", "company_name"] if c in text_model_data.columns),
        None,
    )
    if company_col is None:
        raise ValueError("No company identifier column found.")
    
    # Use only numeric model features. In particular, exclude finbert_text_hash
    # and use the actual Sentence Transformer column prefix text_embedding_.
    base_text_feature_columns = (
        [c for c in text_model_data.columns if c.startswith("text_embedding_")]
        + [
            c for c in [
                "finbert_positive_mean",
                "finbert_neutral_mean",
                "finbert_negative_mean",
                "finbert_negative_max",
                "finbert_negative_std",
                "finbert_positive_minus_negative",
            ]
            if c in text_model_data.columns
        ]
    )
    
    delta_feature_columns = [
        c for c in text_model_data.columns if c.endswith("_delta1")
    ]
    
    loco_feature_columns = list(
        dict.fromkeys(base_text_feature_columns + delta_feature_columns)
    )
    
    # Final safety check: LOCO must never receive non-numeric/hash/string columns.
    loco_feature_columns = [
        c for c in loco_feature_columns
        if pd.api.types.is_numeric_dtype(text_model_data[c])
    ]
    
    if not loco_feature_columns:
        print("Run Sentence Transformer / FinBERT feature generation first.")
    else:
        from sklearn.impute import SimpleImputer
        from sklearn.preprocessing import StandardScaler
        from sklearn.pipeline import Pipeline
        from sklearn.linear_model import LogisticRegression
    
        print("LOCO numeric feature columns:", len(loco_feature_columns))
    
        loco_rows = []
        for company_value in sorted(text_model_data[company_col].dropna().unique()):
            held_out = text_model_data[
                (text_model_data[company_col] == company_value)
                & text_model_data["target_clean"].notna()
            ].copy()
            training_pool = text_model_data[
                (text_model_data[company_col] != company_value)
                & text_model_data["target_clean"].notna()
            ].copy()
    
            if len(held_out) < 2 or training_pool["target_clean"].nunique() < 2:
                continue
    
            pipe = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                ("classifier", LogisticRegression(
                    penalty="l2",
                    C=1.0,
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=5000,
                    random_state=RANDOM_STATE,
                )),
            ])
            pipe.fit(
                training_pool[loco_feature_columns],
                training_pool["target_clean"],
            )
            p = pipe.predict_proba(held_out[loco_feature_columns])[:, 1]
            y = held_out["target_clean"].astype(int)
            pred = (p >= 0.5).astype(int)
    
            row = evaluation_row(
                f"LOCO {company_value}",
                y,
                p,
                pred,
                base_p=float(training_pool["target_clean"].mean()),
            )
            row[company_col] = company_value
            row["n_test_rows"] = len(held_out)
            loco_rows.append(row)
    
        company_generalization_results = pd.DataFrame(loco_rows)
        if not company_generalization_results.empty:
            display(company_generalization_results)
            print("Companies evaluated:", len(company_generalization_results))
